In [3]:
import sys
sys.dont_write_bytecode = True

import warnings
warnings.filterwarnings("ignore")

import importlib
from src.main.python.schema import model
importlib.reload(model)

<module 'src.main.python.schema.model' from 'd:\\dev_space\\LLM-Server\\src\\main\\python\\schema\\model.py'>

# unittest

In [8]:
import sys
import subprocess
!python -B -m unittest src.unittest.python.test_LLMserver

.
----------------------------------------------------------------------
Ran 1 test in 0.074s

OK


# dev

In [1]:
import sys
sys.dont_write_bytecode = True

from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
# PATH = "D://LLM//gemma//gemma3_4b"
PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

In [2]:
import uuid
import importlib
# importlib.reload(scheduler)
_scheduler = scheduler.RequestManager()
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    ids = tokenizer.encode(MSG.format(prompt=sentences))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    _scheduler.add_request(request)

    TEXT = ""
    for _ in range(8):
        d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
        text, DCACHES = decode.infer(llm_model, d_input_ids, d_caches)
        _, PCACHES = prefill.infer(llm_model, p_input_ids, p_caches)
        _scheduler.update(PCACHES)
        if text is not None:
            decode_requests[0].input_ids = torch.tensor(text[0][-1:]).unsqueeze(0)
            decode_requests[0].kv_cache = DCACHES[0]
            _scheduler.add_request(decode_requests[0])
            try:
                TEXT += tokenizer.decode(text[0], skip_special_tokens=True)
            except:
                continue
    print(TEXT)

`cache.key_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].keys` instead.
`cache.value_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].values` instead.


半導體廠務通常在做以下任務：
*   **硬件製造：**
    *   製造各種半導體晶片，包括晶片組件、晶片管、晶片管、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片
、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶片、晶Black Scholes' core spirit is based on statistical knowledge, and it is detailed.
It is also.
is.
is.
is.
is.
is
is.
is.
is
is.

is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
is.
[所有回應一律用繁體中文]

What is the meaning of "
is?
...

is?
*
is?
is?
of?
is?
...
of

is?
...
of

 is?
of?

is?
...
of
 is?
of?

人類是個複雜的系統，包括：
*   **複雜的系統：**
    *   **數據：**
        *   **信息：**
            *   **知識：**
                *   **技能：**
                *   **經驗：**
                *   **


In [4]:
A = torch.tensor([[239102, 238952, 241465, 238659,  30346, 237075, 237893,  26549,    106,
            107,    105,   4368,      0,      0,      0,      0]])
torch.where(A==0)

(tensor([0, 0, 0, 0]), tensor([12, 13, 14, 15]))

In [11]:
p_caches[0].key_cache[0].shape

torch.Size([1, 1, 16, 256])

In [5]:
d_input_ids, ids[-1], d_caches[0].key_cache[0][0][:, :, 47:50, :3]

([tensor([[238786]])],
 107,
 tensor([[[[-0.7812, -0.7812, -1.7188],
           [ 0.2324, -0.0811, -0.3145],
           [ 0.6367,  2.3281,  1.7969]]]], device='cuda:0',
        dtype=torch.bfloat16))

In [4]:
print(TEXT)

半導體廠務通常在做以下任務：
*   **硬件


In [8]:
DCACHES[0].key_cache[0][:, :, :53, :3].shape

torch.Size([1, 1, 45, 3])

In [12]:
cache1.key_cache[0][:, :, 47:53, :3]

tensor([[[[-0.7812, -0.7812, -1.7188],
          [ 3.4062, -0.5898, -0.5664],
          [ 0.2324, -0.0811, -0.3145],
          [ 0.6367,  2.3281,  1.7969],
          [-5.8750,  3.4844, -1.1406],
          [-3.0156,  0.8984, -0.5273]]]], device='cuda:0',
       dtype=torch.bfloat16)

In [18]:
import json
with open("./src/unittest/python/unittest_mock2.json", "w+") as file:
    file.write(json.dumps(cache1.key_cache[0][:, :, 47:53, :3].tolist()))

In [18]:
len(ids)

29

In [9]:
DCACHES[0].key_cache[0][:, :, :17, :]

tensor([[[[ 0.0347, -0.0398, -0.1006,  ..., -2.4844,  0.2891, -3.6094],
          [-2.8125, -0.0664,  1.4375,  ...,  2.3281,  0.0165,  2.6094],
          [-3.2812,  0.3945, -1.2500,  ..., -1.5547,  0.0703, -3.6719],
          ...,
          [-0.5078, -0.5820, -0.0430,  ..., -0.8086,  0.9883, -3.3906],
          [-1.4453, -2.0000,  1.6016,  ..., -2.0156,  0.4648, -2.3281],
          [-0.0469,  0.3633,  0.9258,  ..., -1.8438,  0.6523, -2.9062]]]],
       device='cuda:0', dtype=torch.bfloat16)

In [10]:
cache1.key_cache[0][:, :, :17, :]

tensor([[[[ 0.0347, -0.0398, -0.1006,  ..., -2.4844,  0.2891, -3.6094],
          [-2.8125, -0.0664,  1.4375,  ...,  2.3281,  0.0165,  2.6094],
          [-3.2812,  0.3945, -1.2500,  ..., -1.5547,  0.0703, -3.6719],
          ...,
          [-0.5078, -0.5820, -0.0430,  ..., -0.8086,  0.9883, -3.3906],
          [-1.2188, -0.8984,  1.1328,  ..., -1.1328, -0.3750, -2.0312],
          [ 2.2188,  2.3750, -1.5703,  ...,  2.2969, -0.4180, -4.9062]]]],
       device='cuda:0', dtype=torch.bfloat16)

In [11]:
cache1.key_cache[0].shape

torch.Size([1, 1, 65, 256])

In [7]:
len(ids)

29

In [ ]:
import json
with open("./src/unittest/python/unittest_mock1.json", "w+") as file:
    file.write(json.dumps(cache1.key_cache[0][:, :, :49, :].tolist()))

# test

In [12]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt):
    max_seq_len = 16

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]
    print(len(input_ids[0]))

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    attention_mask = torch.ones(1, ed, dtype=torch.long, device=llm_model.device)
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                # cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)
                cache_position = torch.arange(past_key_values.get_seq_length(layer_idx=0)-1, 
                                              past_key_values.get_seq_length(layer_idx=0), 
                                              dtype=torch.long, 
                                              device = llm_model.device)
                print(cache_position)
                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                # print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [17]:
text1, cache1 = gemma3_resp("半導體廠務通常在做什麼")

29
tensor([27], device='cuda:0')
tensor([28], device='cuda:0')
tensor([29], device='cuda:0')
tensor([30], device='cuda:0')
tensor([31], device='cuda:0')
tensor([32], device='cuda:0')
tensor([33], device='cuda:0')
tensor([34], device='cuda:0')
tensor([35], device='cuda:0')
tensor([36], device='cuda:0')
tensor([37], device='cuda:0')
tensor([38], device='cuda:0')
tensor([39], device='cuda:0')
tensor([40], device='cuda:0')
tensor([41], device='cuda:0')
tensor([42], device='cuda:0')


In [18]:
text1

'半導體廠務通常在做以下幾個核心任務：\n\n1.'

In [ ]:
_, cache1 = gemma3_resp("半導體廠務通常在做什麼")

半導體廠務通常在做以下幾個核心任務：

1.

In [ ]:
_, cache2 = gemma3_resp("什麼是AI ? 一句話介紹一下")

AI (Artificial Intelligence) 是一種人工智能 (Artificial Intelligence) 的一種方法，它通過使用计算机 (c-a-I) 模拟人类的智能，从而学习和模仿人类的语言、动作、学习和推理能力。

AI 是一種高度智能的機器學習 (Machine Learning

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F
from typing import List, Dict, Any
from transformers import DynamicCache

def KVCache_merge(caches: List[DynamicCache]):
    results = DynamicCache()
    # ----- 檢查是否全部都為空 ----- #
    empty_check = [c.key_cache[0] is None for c in caches]
    if np.all(empty_check):
        return results

    # ----- 取出layer ----- #
    seq_len = max(c.get_seq_length(layer_idx=0) for c, is_empty in zip(caches, empty_check) if not is_empty)
    first_non_empty_cache = next(c for c, is_empty in zip(caches, empty_check) if not is_empty)
    n_layers = len(first_non_empty_cache.key_cache)
    n_heads, hid_dim = first_non_empty_cache.key_cache[0].shape[1], first_non_empty_cache.key_cache[0].shape[3]
    
    # ----- 建立cache ----- #
    for i in range(n_layers):
        # ----- 依照不同layer去建立cache ----- #
        keys, values = list(), list()
        for c in caches:
            if c.key_cache[0] is None:
                # ----- 如果是空的，則全部補0 ----- #
                key_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                value_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                keys += [key_tensor]
                values += [value_tensor]
                continue
            key_tensor = c.key_cache[i]
            value_tensor = c.value_cache[i]

            # ----- 過長的部分做padding ----- # 
            curr_seq_len = key_tensor.shape[2]
            if curr_seq_len < seq_len:
                padding_to_add = seq_len - curr_seq_len
                key_tensor = F.pad(key_tensor, (0, 0, padding_to_add, 0), "constant", 0)
                value_tensor = F.pad(value_tensor, (0, 0, padding_to_add, 0), "constant", 0)

            keys += [key_tensor]
            values += [value_tensor]
        
        # ----- merge tensor ----- #
        key_batch = torch.cat(keys, dim=0)
        value_batch = torch.cat(values, dim=0)

        # ----- update cache ----- #
        results.update(key_states=key_batch, value_states=value_batch, layer_idx=i)
    results.seen_tokens = seq_len
    return results

In [ ]:
CACHE = KVCache_merge([cache1, cache2])

In [ ]:
CACHE.get_seq_length(layer_idx=0)

169

In [ ]:
CACHE.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],


In [ ]:
def KVCache_split(cache: DynamicCache):
    # ----- 把cache的layer跟數量定義出來 ----- #
    batch_size = cache.key_cache[0].shape[0]
    n_layers = len(cache.key_cache)

    # ----- return的結果 ----- #
    results: List[DynamicCache] = [DynamicCache() for _ in range(batch_size)]

    # ----- by batch操作
    for i in range(batch_size):
        # ----- cache的原始長度，只要用第0層來找即可 ----- #
        """
        因為padding是用0填充，所以如果 hid_dim 和 n_head 都是0，那該位必定padding
        最終找到最後一個非零位置
        """
        sample_key_tensor = cache.key_cache[0][i:i+1] # (1, n_heads, seq_len, hid_dim)
        sum_abs = torch.abs(sample_key_tensor).sum(dim=(1, 3)).squeeze(0)
        non_zero_indices = torch.where(sum_abs > 1e-6)[0] # (seq_len, )

        # ----- seq_len 儲存長度計算 ----- #
        if len(non_zero_indices) == 0: original_seq_len = 0
        else: original_seq_len = non_zero_indices.min().item()
            
        for layer_idx in range(n_layers):
            key_slice = cache.key_cache[layer_idx][i:i+1]
            value_slice = cache.value_cache[layer_idx][i:i+1]

            truncated_key = key_slice[:, :, original_seq_len:, :]
            truncated_value = value_slice[:, :, original_seq_len:, :]
            
            results[i].update(
                key_states=truncated_key,
                value_states=truncated_value,
                layer_idx=layer_idx
            )
        
        # 6. 更新這個 cache 的 seen_tokens
        results[i].seen_tokens = original_seq_len

    return results

In [ ]:
_cache1, _cache2 = KVCache_split(CACHE)
_cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0347],
          [-2.8125],
          [-3.2812],
          [-0.4941],
          [ 0.1016],
          [ 1.5000],
          [ 0.8594],
          [-4.0625],
          [-3.6094],
          [-1.4531],
          [ 3.5625],
          [ 4.5312],
          [ 1.5156],
          [-1.7109],
          [-0.5078],
          [-2.1250],
          [-0.5391],
          [-0.0435],
          [ 2.9062],
          [-0.9453],
          [ 0.1157],
          [-0.4453],
          [-0.0967],
          [ 2.5156],
          [ 0.2139],
          [ 0.7148],
          [-2.4219],
          [-3.1719],
          [-1.1016],
          [ 1.7188],
          [ 3.3906],
          [ 1.2656],
          [-1.9922],
          [-0.1641]]]], device='cuda:0', dtype=torch.bfloat16)

In [ ]:
cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0347],
          [-2.8125],
          [-3.2812],
          [-0.4941],
          [ 0.1016],
          [ 1.5000],
          [ 0.8594],
          [-4.0625],
          [-3.6094],
          [-1.4531],
          [ 3.5625],
          [ 4.5312],
          [ 1.5156],
          [-1.7109],
          [-0.5078],
          [-2.1250],
          [-0.5391],
          [-0.0435],
          [ 2.9062],
          [-0.9453],
          [ 0.1157],
          [-0.4453],
          [-0.0967],
          [ 2.5156],
          [ 0.2139],
          [ 0.7148],
          [-2.4219],
          [-3.1719],
          [-1.1016],
          [ 1.7188],
          [ 3.3906],
          [ 1.2656],
          [-1.9922],
          [-0.1641]]]], device='cuda:0', dtype=torch.bfloat16)

In [5]:
import torch
tensor=torch.tensor([1,2,3,4,0,0,0])
where = torch.where(tensor==0)[0]
tensor[:(where - len(tensor))[0]]

tensor([1, 2, 3, 4])

In [6]:
tensor.unsqueeze(1)

tensor([[1],
        [2],
        [3],
        [4],
        [0],
        [0],
        [0]])